# Block on an inclined plane with friction

*mechanics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/inclined_plane)

Gravity, the normal force and friction on a block, resolved along each ramp's slope and normal unit vectors with einsum projections. A batch of incline angles shows the static-friction threshold $\tan\theta = \mu_s$.

**Vector form**

$$m\,\mathbf a = m\,\mathbf g + \mathbf N + \mathbf f$$

**Index (tensor) form**

$$a_{bi} = g_i + \tfrac{1}{m}N_{bi} + \tfrac{1}{m}f_{bi},\qquad \tfrac{1}{m}N_{bi} = -(g_j n_{bj})\,n_{bi},\qquad \tfrac{1}{m}f_{bi} = -\mu_k\,\tfrac{1}{m}N_b\,s_{bi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $s_{bi}$ | `s` | (B, 2) | unit vector down the slope of ramp $b$, $(\cos\theta, -\sin\theta)$ |
| $n_{bi}$ | `n` | (B, 2) | unit normal of ramp $b$, $(\sin\theta, \cos\theta)$ |
| $g_i$ | `g` | (2,) | gravity vector $(0, -g)$ |
| $g_j n_{bj}$ | `einsum('i,bi->b', g, n)` | (B,) | repeated index ⇒ summed: component of gravity into the ramp |
| $N_{bi}/m$ | `normal_force(g, n)` | (B, 2) | normal force per unit mass, cancels the into-ramp part of gravity |
| $f_{bi}/m$ | `friction(g, s, n, mu_s, mu_k)` | (B, 2) | static friction if it can hold the block, otherwise kinetic $\mu_k N$ |
| $a_{bi}s_{bi}$ | `einsum('bi,bi->b', a, s)` | (B,) | acceleration along the slope: $g(\sin\theta - \mu_k\cos\theta)$ or 0 |

Static friction is as large as it needs to be, up to $\mu_s N$, so a block released from rest stays put while $\tan\theta \le \mu_s$. Once it slides, kinetic friction $\mu_k N$ (with $\mu_k \le \mu_s$) takes over and the acceleration is constant, so the distance grows as $\tfrac12 at^2$ until the block reaches the bottom. The block is treated as a point: no tipping, no rolling.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/inclined_plane/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    mu_s, mu_k = params["mu_static"], max(0.0, min(params["mu_kinetic"], params["mu_static"]))  # μ_k ≤ μ_s
    g = tensors.gravity(params["g"], dev)
    theta = tensors.ramp_angles(params["n_ramps"], params["min_angle"], params["max_angle"])
    s, n = tensors.unit_vectors(theta, dev)

    a = equation.acceleration(g, s, n, mu_s, mu_k)  # (B, 2)
    sol = solve.solve(a, s, params["ramp_length"], params["t_max"])

    print(f"block stays at rest when θ < arctan(μ_s) = {math.degrees(math.atan(mu_s)):.1f}°")
    for deg, a_par in zip(theta.rad2deg().tolist(), sol["a_par"].tolist()):
        state = "slides" if a_par > 1e-6 else "at rest"
        print(f"angle {deg:5.1f}°  a along slope {a_par:6.3f} m/s²  ({state})")
    return plot.animate(sol, theta, params["ramp_length"], out_dir / "inclined_plane.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/inclined_plane/equation.py
r"""Physics → tensor translation.

A block of mass m on a ramp feels three forces: gravity, the normal force N from the ramp and friction f.
Newton's second law, divided by m:

    \mathbf a = \mathbf g + \mathbf N / m + \mathbf f / m

For a batch of ramps b with angles θ_b we use two unit vectors per ramp (components i ∈ {x, z}):

    s_{bi} = (\cos\theta_b, -\sin\theta_b)     points down the slope
    n_{bi} = (\sin\theta_b,  \cos\theta_b)     points out of the surface

Projecting a vector onto a unit vector is a repeated (summed) index, einsum('i,bi->b'):

    g_{\parallel b} = g_i s_{bi} = g \sin\theta_b,      g_{\perp b} = g_i n_{bi} = -g \cos\theta_b

The ramp pushes back just enough to cancel the normal part of gravity, and friction acts along the slope:

    N_{bi}/m = -g_{\perp b} \, n_{bi}
    f_{bi}/m = -\min(g_{\parallel b}, \mu_s N_b/m) \, s_{bi}       if the block can stay at rest (static)
    f_{bi}/m = -\mu_k (N_b/m) \, s_{bi}                             if it slides (kinetic)

    a_{bi} = g_i + N_{bi}/m + f_{bi}/m

The block stays put when g_∥ ≤ μ_s N/m, i.e. tan θ ≤ μ_s. Otherwise a_∥ = g (sin θ − μ_k cos θ).
"""

import torch


def project(vector: torch.Tensor, unit: torch.Tensor) -> torch.Tensor:
    r"""Component along each ramp's unit vector:  v_i u_{bi}  (i summed).   (2,), (B, 2) → (B,)"""
    return torch.einsum("i,bi->b", vector, unit)


def normal_force(g: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""N_{bi}/m = -(g_j n_{bj}) n_{bi}: cancels the part of gravity pushing into the ramp.   → (B, 2)"""
    g_perp = project(g, n)  # (B,)  = -g cos θ
    return -torch.einsum("b,bi->bi", g_perp, n)


def friction(g: torch.Tensor, s: torch.Tensor, n: torch.Tensor, mu_s: float, mu_k: float) -> torch.Tensor:
    r"""f_{bi}/m along -s_{bi}: static if it can hold the block, kinetic otherwise.   → (B, 2)"""
    g_par = project(g, s)  # (B,)  pull down the slope, g sin θ
    normal = -project(g, n)  # (B,)  size of N/m, g cos θ
    stuck = g_par <= mu_s * normal  # (B,) True where static friction wins
    size = torch.where(stuck, g_par, mu_k * normal)  # (B,)
    return -torch.einsum("b,bi->bi", size, s)


def acceleration(g: torch.Tensor, s: torch.Tensor, n: torch.Tensor, mu_s: float, mu_k: float) -> torch.Tensor:
    r"""a_{bi} = g_i + N_{bi}/m + f_{bi}/m.   → (B, 2)"""
    return g + normal_force(g, n) + friction(g, s, n, mu_s, mu_k)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/inclined_plane/tensors.py
"""Build the concrete tensors: gravity, the ramp angles and each ramp's unit vectors."""

import torch


def gravity(g: float, device: torch.device) -> torch.Tensor:
    """g_i with i ∈ {x, z}: shape (2,)."""
    return torch.tensor([0.0, -g], device=device)


def ramp_angles(n: int, min_deg: float, max_deg: float) -> torch.Tensor:
    """n incline angles from min_deg to max_deg (or just max_deg for a single ramp), in radians: (B,)."""
    if n == 1:
        return torch.deg2rad(torch.tensor([max_deg]))
    return torch.deg2rad(torch.linspace(min_deg, max_deg, n))


def unit_vectors(theta: torch.Tensor, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Down-slope s_bi and surface normal n_bi for every ramp: two tensors of shape (B, 2).

    s[b] = ( cos θ_b, -sin θ_b)
    n[b] = ( sin θ_b,  cos θ_b)
    """
    theta = theta.to(device)
    c, s = torch.cos(theta), torch.sin(theta)
    down_slope = torch.stack([c, -s], dim=-1)
    normal = torch.stack([s, c], dim=-1)
    return down_slope, normal


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/inclined_plane/solve.py
"""Move every block down its ramp. The forces never change, so the acceleration is constant."""

import torch


def solve(a: torch.Tensor, s: torch.Tensor, ramp_length: float, t_max: float, n_times: int = 301) -> dict:
    """Return times (T,), distance travelled along the slope d (T, B) and speed v (T, B).

    The block starts at rest at the top. With constant acceleration a_{bi},
    the distance along the slope is d_b(t) = ½ a_∥b t², with a_∥b = a_{bi} s_{bi}.
    A block that reaches the bottom of the ramp stops there.
    """
    a_par = torch.einsum("bi,bi->b", a, s)  # (B,) acceleration along the slope
    times = torch.linspace(0.0, t_max, n_times, device=a.device)  # (T,)
    d = 0.5 * torch.einsum("t,b->tb", times**2, a_par)  # (T, B)
    v = torch.einsum("t,b->tb", times, a_par)
    at_bottom = d >= ramp_length
    d = torch.where(at_bottom, ramp_length, d)
    v = torch.where(at_bottom, 0.0, v)
    return {"times": times, "d": d, "v": v, "a_par": a_par}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/inclined_plane/plot.py
"""Animate one block per ramp, side by side, with the distance slid along each slope below."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Polygon

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(sol: dict, theta: torch.Tensor, ramp_length: float, path: Path) -> Path:
    t = sol["times"].cpu().numpy()  # (T,)
    d = sol["d"].cpu().numpy()  # (T, B)
    a_par = sol["a_par"].cpu().numpy()  # (B,)
    angle = theta.cpu().numpy()  # (B,)
    n_ramps = len(angle)
    L, size = ramp_length, 0.12 * ramp_length  # ramp length and block edge

    s = np.stack([np.cos(angle), -np.sin(angle)], axis=-1)  # (B, 2) down the slope
    n = np.stack([np.sin(angle), np.cos(angle)], axis=-1)  # (B, 2) out of the surface
    widths = L * np.cos(angle)
    left = np.concatenate([[0.0], np.cumsum(widths + 0.25 * L)[:-1]])  # x of each ramp's top corner
    top = np.stack([left, L * np.sin(angle)], axis=-1)  # (B, 2) where each block starts

    fig, (ax_ramp, ax_dist) = plt.subplots(2, 1, figsize=(7, 4.6), gridspec_kw={"height_ratios": [1.3, 1]})

    # --- top: the ramps and blocks ---
    ax_ramp.set_xlim(-0.1 * L, left[-1] + widths[-1] + 0.1 * L)
    ax_ramp.set_ylim(-0.25 * L, L * np.sin(angle).max() + 0.3 * L)
    ax_ramp.set_aspect("equal", adjustable="box")
    ax_ramp.axis("off")
    blocks = []
    for b in range(n_ramps):
        color = ACCENT[b % len(ACCENT)]
        corners = [top[b], top[b] + L * s[b], [left[b], 0.0]]  # right-angled triangle
        ax_ramp.add_patch(Polygon(corners, closed=True, color="#e5e7eb", ec="#6b7280"))
        state = "slides" if a_par[b] > 1e-6 else "at rest"
        ax_ramp.text(
            left[b] + 0.4 * widths[b], -0.2 * L, f"{np.degrees(angle[b]):.0f}°  {state}", fontsize=7, ha="center"
        )
        blocks.append(ax_ramp.add_patch(Polygon(np.zeros((4, 2)), closed=True, color=color)))
    clock = ax_ramp.text(0.0, 1.0, "", transform=ax_ramp.transAxes, va="top", fontsize=9)

    # --- bottom: distance along the slope vs time ---
    ax_dist.set_xlim(t[0], t[-1])
    ax_dist.set_ylim(0, L * 1.08)
    ax_dist.axhline(L, color="#9ca3af", lw=0.8, ls="--")
    ax_dist.set_xlabel("t (s)")
    ax_dist.set_ylabel("distance slid (m)")
    curves = [ax_dist.plot([], [], lw=1.6, color=ACCENT[b % len(ACCENT)])[0] for b in range(n_ramps)]

    def update(step: int):
        for b in range(n_ramps):
            centre = top[b] + d[step, b] * s[b] + 0.5 * size * n[b]
            half_s, half_n = 0.5 * size * s[b], 0.5 * size * n[b]
            blocks[b].set_xy(
                [centre - half_s - half_n, centre + half_s - half_n, centre + half_s + half_n, centre - half_s + half_n]
            )
            curves[b].set_data(t[: step + 1], d[: step + 1, b])
        clock.set_text(f"t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'mu_static': 0.5,
    'mu_kinetic': 0.35,
    'n_ramps': 5,
    'min_angle': 15.0,
    'max_angle': 50.0,
    'ramp_length': 4.0,
    'g': 9.81,
    't_max': 2.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.inclined_plane")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.inclined_plane.main")
problem_dir = Path("problems/mechanics/inclined_plane")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
